In [13]:
import torch
import torch.nn as nn

In [14]:
class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, padding=1),
            nn.ReLU()              
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=3, padding=1),
            nn.ReLU()
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)

        return x

In [15]:
class DecoderBlock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.convt2d = nn.ConvTranspose2d(in_channels=in_channels, out_channels=in_channels//2, kernel_size=2, stride=2)
        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=in_channels//2, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(in_channels=in_channels//2, out_channels=in_channels//2, kernel_size=3,padding=1),
            nn.ReLU()
        )

    def forward(self, x, skips):
        x = self.convt2d(x)
        x = torch.concat((x, skips), dim=1)
        x = self.layer1(x)
        x = self.layer2(x)

        return x


In [16]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.pool = nn.MaxPool2d(2, 2)

        self.enc1 = EncoderBlock(1, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 256)
        self.enc4 = EncoderBlock(256, 512)
        self.bottleneck = EncoderBlock(512, 1024)

        self.dec1 = DecoderBlock(1024)
        self.dec2 = DecoderBlock(512)
        self.dec3 = DecoderBlock(256)
        self.dec4 = DecoderBlock(128)

        self.out_conv = nn.Conv2d(64, 1, 1)
    
    def forward(self, x):
        x = self.enc1(x)
        skip1 = x
        x = self.pool(x)

        x = self.enc2(x)
        skip2 = x
        x = self.pool(x)

        x = self.enc3(x)
        skip3 = x
        x = self.pool(x)

        x = self.enc4(x)
        skip4 = x
        x = self.pool(x)

        x = self.bottleneck(x)
        
        x = self.dec1(x, skip4)
        x = self.dec2(x, skip3)
        x = self.dec3(x, skip2)
        x = self.dec4(x, skip1)

        x = self.out_conv(x)

        return x


In [17]:
model = UNet()
x = torch.randn(1, 1, 256, 256)
out = model(x)
print(out.shape)

torch.Size([1, 1, 256, 256])
